In [70]:
import os
import pandas as pd
from pystac_client import Client
import pandas as pd

## Connect to Montandon STAC

In [8]:
STAC_API_URL = "https://montandon-eoapi.ifrc.org/stac"
API_TOKEN = os.getenv('MONTANDON_API_TOKEN')

In [11]:
auth_headers = {"Authorization": f"Bearer {API_TOKEN}"}

### Check Auth

In [17]:
# Connect to STAC API with authentication
try:
    client = Client.open(STAC_API_URL, headers=auth_headers)
    print(f"\n[OK] Connected to: {STAC_API_URL}")
    print(f"[OK] API Title: {client.title}")
    print(f"[OK] Authentication: Bearer Token (OpenID Connect)")
except Exception as e:
    print(f"\n[ERROR] Authentication failed: {e}")


[OK] Connected to: https://montandon-eoapi.ifrc.org/stac
[OK] API Title: Montandon STAC API
[OK] Authentication: Bearer Token (OpenID Connect)


In [15]:
client

<Client id=montandon-eoapi>

## Fetch most recent Events

List collections with event items, usig simple string-matching on the collection id

In [22]:
collections_list = list(client.get_collections())

In [24]:
events_collection = [c for c in collections_list if '-events' in c.id]

In [33]:
for col in events_collection:
    print(f"{col.title} | {col.id}")

DesInventar Mapped Events | desinventar-events
EM-DAT Source Events | emdat-events
GDACS Source Events | gdacs-events
GFD Source Events | gfd-events
GLIDE Source Events | glide-events
IBTrACS Source Events | ibtracs-events
IDMC GIDD Source Events | idmc-gidd-events
IDMC Internal Displacement Updates (IDU) Impacts | idmc-idu-events
IFRC Source Events | ifrcevent-events
PDC Source Events | pdc-events
USGS Events | usgs-events


### Single Collection Search
Search a single collection for the most earthquakes from USGS that have a magnitude > 5

In [111]:
search = client.search(
    collections=["usgs-events"],
    max_items=1000
)
items = list(search.items())

In [112]:
# Get the shape of an item
items[0]

<Item id=usgs-event-ci41469920>

In [113]:
# Build dataframe, sort by chronological order
df = pd.DataFrame(
    [{
        'id': item.id,
        'datetime': pd.to_datetime(item.properties['datetime']),
        'title': item.properties['title'],
        'magnitude': item.properties['eq:magnitude'],
        'geometry': item.geometry['coordinates']
    } for item in items
    ]
).sort_values('datetime', ascending=False)

In [114]:
df[df['magnitude'] > 5]

,id,datetime,title,magnitude,geometry
54,usgs-event-us6000sykw,2026-05-18 18:50:00.309000+00:00,M 5.3 - Bouvet Island region,5.3,"[-1.2672, -54.2795]"
116,usgs-event-us6000syik,2026-05-18 13:44:26.230000+00:00,"M 5.1 - 26 km NW of Liuzhou, China",5.1,"[109.2073, 24.4799]"
215,usgs-event-us6000syh2,2026-05-18 07:34:30.448000+00:00,M 5.2 - Mid-Indian Ridge,5.2,"[74.2599, -28.5299]"
254,usgs-event-us6000syg1,2026-05-18 02:10:41.745000+00:00,M 5.4 - West Chile Rise,5.4,"[-82.4593, -43.4009]"
255,usgs-event-us6000syg0,2026-05-18 02:05:24.327000+00:00,"M 5.2 - 36 km SSE of Syriam, Burma (Myanmar)",5.2,"[96.3164, 16.4431]"
411,usgs-event-us6000syc2,2026-05-17 08:04:17.461000+00:00,"M 5.4 - 84 km ENE of Severo-Kuril’sk, Russia",5.4,"[157.2919, 50.8582]"
413,usgs-event-us6000syby,2026-05-17 07:51:56.627000+00:00,"M 5.1 - 109 km SSE of Lorengau, Papua New Guinea",5.1,"[147.7681, -2.8876]"
547,usgs-event-us6000sy90,2026-05-16 15:58:04.288000+00:00,"M 5.4 - 92 km E of Port-Vila, Vanuatu",5.4,"[169.1752, -17.8726]"
556,usgs-event-us6000sy8k,2026-05-16 15:17:56.098000+00:00,"M 5.7 - 33 km NE of Auki, Solomon Islands",5.7,"[160.9161, -8.5526]"
558,usgs-event-us6000sy84,2026-05-16 14:50:03.503000+00:00,"M 6.0 - 70 km ESE of Codrington, Antigua and B...",6.0,"[-61.1771, 17.5127]"
